In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l6.8 Ablation harness

> 🎯 **Goal:** Build a small harness that trains each architecture variant under
identical conditions and tabulates the loss deltas, so the choices are evidence, not
folklore.

**Why this matters.** This lesson ties the whole unit together. You added RoPE,
RMSNorm, and SwiGLU on the promise that they help, but a promise is not proof. The
ablation harness is how you actually earn the modernized PocketLM: train the baseline
and each upgrade head to head and read off which ones paid.

**The intuition.** This is the scientific method applied to architecture. To learn
what a change does, hold everything else fixed (same data, same seed, same steps, same
optimizer) and toggle exactly one thing at a time. The difference in the final loss,
the *delta*, is your measurement of that one change. Change two things at once and you
cannot tell which one moved the number.

**The mechanics.** The `run` helper takes config overrides, builds a model, trains it
for a fixed number of steps from a fixed seed, and returns the validation loss. We
then define a dictionary of variants (baseline, plus RMSNorm, plus SwiGLU, and the
fully modern stack) and run each one. Because every run starts from the same seed and
sees the same batches, the only thing that differs between two rows is the override
we changed, so the loss gap is attributable to that override alone.

In [ ]:
from pocketlm import (CharTokenizer, PocketLM, PocketLMConfig, init_weights,
                      make_batch, estimate_loss)

corpus = open(CORPUS_PATH, encoding="utf-8").read()
tok = CharTokenizer.from_text(corpus)
data = torch.tensor(tok.encode(corpus), dtype=torch.long)

def run(**overrides):
    torch.manual_seed(0)
    cfg = PocketLMConfig(vocab_size=tok.vocab_size, block_size=32, n_layer=2,
                         n_head=4, n_embd=64, **overrides)
    model = init_weights(PocketLM(cfg))
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    g = torch.Generator().manual_seed(0)
    steps = 120 if os.environ.get("POCKETLM_CI") else 400
    for _ in range(steps):
        x, y = make_batch(data, 32, 16, generator=g)
        _, loss = model(x, y)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
    return estimate_loss(model, data, 32, 16, iters=10, generator=g)

variants = {
    "baseline (LN+GELU+learned)": {},
    "+ rmsnorm": dict(norm="rmsnorm"),
    "+ swiglu": dict(mlp="swiglu"),
    "modern (rms+swiglu+rope)": dict(norm="rmsnorm", mlp="swiglu", pos="rope"),
}
results = {name: run(**cfg) for name, cfg in variants.items()}
for name, loss in results.items():
    print(f"{name:<28} val loss {loss:.3f}")

**What just happened.** The table lists the validation loss for each variant under
identical training. Reading down the rows, you can attribute each loss change to the
one override that produced it, and the bottom row (the fully modern stack) shows what
the combined upgrades buy. The choices are now evidence-based rather than folklore.

⚠️ **Common pitfalls**
- One seed is one data point. At this toy scale the differences can be within noise; a
  real harness averages over several seeds and reports the variance before declaring a
  winner.
- Changing more than one knob per row. If a variant overrides two things, you cannot
  separate their effects; keep each ablation to a single change from the baseline.
- Comparing across different training conditions. If two runs saw different data,
  steps, or learning rates, the loss gap is not a clean measurement of the architecture.

🏋️ **Try it yourself.** Add a `+ rope` row (`dict(pos="rope")`) so RoPE is ablated on
its own, then run the dictionary again. Does RoPE alone help, and does it add to the
gains from RMSNorm and SwiGLU when all three are combined? That is exactly the
question an ablation is built to answer.

In [ ]:
import math
assert len(results) == 4
assert all(loss < math.log(tok.vocab_size) for loss in results.values())